# 🤖 Module 3 · Class 6 — Olist Capstone (AI Prompt Edition)

**The new way to work:** you do **not** write Python from memory, and you do **not** copy blanks.
For each step you **write a clear prompt**, an **AI** (ChatGPT / Claude / Gemini) writes the code,
and **you paste it, run it, and check the result**.

**Why?** Writing good prompts is a real job skill. By the end of this lab you will know how to make
an AI build a full data pipeline for you — and how to *check* that it did it right.

**Your final product:** the file `olist_clean.parquet` (Module 4 will train on it).

**Level:** A2 English. Solo or in pairs.

## 📦 The dataset you are working with

**Olist** = a real online shop in Brazil. The data is **8 tables (CSV files)**:

| Table (variable) | What is inside | Key columns |
| --- | --- | --- |
| `orders` | each order + its dates | order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date |
| `customers` | who bought + location | customer_id, customer_unique_id, customer_state, customer_zip_code_prefix |
| `items` | products in each order | order_id, order_item_id, seller_id, price, freight_value |
| `payments` | how it was paid | order_id, payment_type, payment_installments, payment_value |
| `reviews` | rating + comment | order_id, review_score, review_comment_message, review_creation_date |
| `sellers` | the seller + location | seller_id, seller_state, seller_zip_code_prefix |
| `geo` | zip code → map point | geolocation_zip_code_prefix, geolocation_lat, geolocation_lng |
| `products` | product details (barely used) | product_id, ... |

**The job:** join the useful pieces into ONE table (one row per order) and build the target
**`is_late`** (1 = the order arrived after the estimated date, 0 = on time).

> You will give the AI these table + column names in your prompts. That is why they are listed here.

## 🧠 The most important skill: the PROMPT RECIPE

A good prompt has **4 parts**. If you miss one, the AI guesses wrong.

| Part | What you write |
| --- | --- |
| **1. Context** | the dataset + which tables/columns you have |
| **2. Goal** | what you need this step to produce |
| **3. What you already have** | the variables that exist from earlier steps |
| **4. Ask** | "write pandas code, comment each line, and tell me the expected result" |

### 📋 Copy this template — reuse it for EVERY stage
```
I'm working in Python with pandas on the Olist Brazilian e-commerce dataset.
These DataFrames are already loaded: orders, customers, items, payments, reviews, sellers, geo.

GOAL (this step): <write what you need to produce>
DETAILS: <the exact columns / any filter / how to combine>
WHAT I ALREADY HAVE: <variables made in earlier steps>

Please write the pandas code, add a short comment on each line, and then tell me
what shape or number I should expect so I can check it.
```

### ❌ Bad prompt → *"write code to clean olist data"* (AI has no idea what you mean)
### ✅ Good prompt → fills in all 4 parts with the real table and column names.

**Every stage below tells you exactly what to put in GOAL and DETAILS. You write the prompt,
paste the AI's code into the empty cell, run it, and check the result.**

## Stage 1 — Load the 8 tables

📋 **Your task:** load all 8 CSV files into DataFrames named `orders, customers, items, payments,
reviews, products, sellers, geo`.

📦 **Give the AI these facts in your prompt:**
- The files are online. The base address is:
  `https://huggingface.co/datasets/aviahYadler/Olist_Ecommerce_Dataset/resolve/main`
- The file names are like `olist_orders_dataset.csv`, `olist_customers_dataset.csv`, … (one per table).
- You want to read each one with `pd.read_csv(BASE + '/' + filename)`.
- After loading, print each table's `.shape`.

✍️ **Write your prompt** (double-click to type), then paste the AI's code in the cell below and run it.

In [ ]:
# 👇 Paste the code the AI gave you here, then run it (Shift+Enter).
import pandas as pd, numpy as np
BASE = "https://huggingface.co/datasets/aviahYadler/Olist_Ecommerce_Dataset/resolve/main"

# Load each CSV directly from HuggingFace into a named DataFrame
orders    = pd.read_csv(BASE + "/olist_orders_dataset.csv")           # order headers + status + timestamps
customers = pd.read_csv(BASE + "/olist_customers_dataset.csv")        # customer IDs + location
items     = pd.read_csv(BASE + "/olist_order_items_dataset.csv")      # line items: price, freight, seller
payments  = pd.read_csv(BASE + "/olist_order_payments_dataset.csv")   # payment type + installments + value
reviews   = pd.read_csv(BASE + "/olist_order_reviews_dataset.csv")    # review scores + comments
products  = pd.read_csv(BASE + "/olist_products_dataset.csv")         # product metadata + category
sellers   = pd.read_csv(BASE + "/olist_sellers_dataset.csv")          # seller IDs + location
geo       = pd.read_csv(BASE + "/olist_geolocation_dataset.csv")      # zip code → lat/lng mappings

# Print shape of every table so you can verify the load
for name, df in [("orders", orders), ("customers", customers), ("items", items),
                 ("payments", payments), ("reviews", reviews), ("products", products),
                 ("sellers", sellers), ("geo", geo)]:
    print(f"{name:12s}: {df.shape}")




✅ **Check your result:** you should see `orders: (99441, 8)`, `geo: (1000163, 5)`, and 6 more.
If the AI's code errors, **paste the error back to it** and ask it to fix it (that is part 4 of the recipe).

## Stage 2 — Clean the `orders` table

📋 **Your task:** make `orders` ready: real dates, only useful orders, no missing delivery dates.

📦 **Put this in your prompt (GOAL + DETAILS):**
- Turn these columns into real dates: `order_purchase_timestamp`, `order_approved_at`,
  `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`
  (use `pd.to_datetime(..., errors='coerce')`).
- Keep only rows where `order_status` is one of `delivered, invoiced, shipped, approved` → call it `orders_model`.
- Drop rows in `orders_model` that have no `order_delivered_customer_date` or no `order_estimated_delivery_date`.
- Print how many rows remain.

✍️ Write the prompt, paste the code below, run it.

In [ ]:
# 👇 Paste the AI's code for Stage 2 here.
# ── 1. Parse all 5 timestamp columns in-place ───────────────────────────────
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")  # unparseable → NaT

# ── 2. Keep only orders that are commercially useful ────────────────────────
useful_statuses = {"delivered", "invoiced", "shipped", "approved"}          # set for O(1) lookup
orders_model = orders[orders["order_status"].isin(useful_statuses)].copy()  # copy avoids SettingWithCopyWarning

# ── 3. Drop rows missing either critical delivery date ───────────────────────
orders_model.dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"],
    inplace=True,   # modify orders_model directly
)

# ── 4. Report ────────────────────────────────────────────────────────────────
print(f"orders_model shape: {orders_model.shape}")   # rows × columns
print(f"Row count          : {len(orders_model)}")   # just the number


✅ **Check:** about **96,470** rows remain (roughly 1,431 dropped). If your number is very different,
tell the AI what you got and ask why.

## Stage 3 — One row per order (aggregate items + payments)

📋 **Your task:** items and payments have many rows per order. Squeeze them to one row per order.

📦 **Put this in your prompt:**
- From `items`: per `order_id`, make `num_items` (count of `order_item_id`), `total_price` (sum of `price`),
  `total_freight` (sum of `freight_value`). Call it `order_totals`.
- From `payments`: keep the **biggest** payment per `order_id` (its `payment_type`, `payment_installments`).
  Call it `primary_pay`.
- From `items`: the **first** seller per `order_id` (`order_id`, `seller_id`). Call it `first_seller`.

✍️ Write the prompt, paste the code, run it.

In [ ]:
# 👇 Paste the AI's code for Stage 3 here.
# ── 1. order_totals: aggregate items → one row per order ────────────────────
order_totals = (
    items.groupby("order_id")           # group by order
    .agg(
        num_items      = ("order_item_id", "count"),   # how many line-items
        total_price    = ("price",         "sum"),      # sum of item prices
        total_freight  = ("freight_value", "sum"),      # sum of freight charges
    )
    .reset_index()                       # bring order_id back as a plain column
)

# ── 2. primary_pay: biggest payment per order ───────────────────────────────
primary_pay = (
    payments
    .sort_values("payment_value", ascending=False)   # largest payment floats to top
    .groupby("order_id", as_index=False)             # one group per order
    .first()                                         # grab the top row (highest value)
    [["order_id", "payment_type", "payment_installments"]]  # keep only needed cols
)

# ── 3. first_seller: earliest line-item seller per order ────────────────────
first_seller = (
    items.sort_values("order_item_id")               # item 1 before item 2, etc.
    .groupby("order_id", as_index=False)             # one group per order
    .first()                                         # grab item_id == 1 row
    [["order_id", "seller_id"]]                      # keep only needed cols
)

# ── 4. Verify shapes ─────────────────────────────────────────────────────────
print(f"order_totals : {order_totals.shape}")
print(f"primary_pay  : {primary_pay.shape}")
print(f"first_seller : {first_seller.shape}")


✅ **Check:** `order_totals` ≈ (98666, 4), `primary_pay` ≈ (99440, 3), `first_seller` ≈ (98666, 2).

## Stage 4 — Distance from seller to customer

📋 **Your task:** build a tool that gives the distance in km between two map points, and a lookup
that turns a zip code into one map point.

📦 **Put this in your prompt:**
- Write a `haversine(lat1, lon1, lat2, lon2)` function that returns the distance in **km** (Earth radius 6371).
- From `geo`: one **average** `lat`/`lon` per `geolocation_zip_code_prefix`. Call it `geo_lookup`.

✍️ Write the prompt, paste the code, run it.

> 💡 The haversine math is tricky — this is a perfect thing to let the AI write. Just **check** the result looks right.

In [ ]:
# 👇 Paste the AI's code for Stage 4 here.
import numpy as np  # needed for trig functions

# ── 1. haversine: great-circle distance between two lat/lon points ───────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371                                      # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])  # degrees → radians
    dlat = lat2 - lat1                            # difference in latitude
    dlon = lon2 - lon1                            # difference in longitude
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2  # haversine formula
    return R * 2 * np.arcsin(np.sqrt(a))          # central angle → km

# ── 2. geo_lookup: one representative lat/lon per zip prefix ────────────────
geo_lookup = (
    geo.groupby("geolocation_zip_code_prefix")    # one group per 5-digit zip prefix
    .agg(
        lat = ("geolocation_lat", "mean"),         # average latitude for that zip
        lon = ("geolocation_lng", "mean"),         # average longitude for that zip
    )
    .reset_index()                                 # zip prefix back as plain column
)

# ── 3. Verify ────────────────────────────────────────────────────────────────
print(f"geo_lookup shape : {geo_lookup.shape}")

# ── 4. Sanity-check haversine with two known Brazilian cities ────────────────
# São Paulo (-23.55, -46.63)  →  Rio de Janeiro (-22.91, -43.17) ≈ 357 km
test_km = haversine(-23.55, -46.63, -22.91, -43.17)
print(f"SP → RJ test     : {test_km:.1f} km  (expect ≈ 357)")


✅ **Check:** `geo_lookup` ≈ (19015, 3) — one row per zip code.

## Stage 5 — Glue everything + build the features (the big one)

📋 **Your task:** start from `orders_model`, merge everything, compute distance, dates, and `is_late`.

📦 **Put this in your prompt (this one is long — give ALL of it):**
- Start with `df = orders_model`. Merge on, using `how='left'`:
  `customers` (on `customer_id`), `order_totals` (`order_id`), `primary_pay` (`order_id`),
  `first_seller` (`order_id`), `sellers` (`seller_id`).
- Add customer lat/lon and seller lat/lon by merging `geo_lookup` on the two zip-code columns.
- `distance_km` = `haversine(cust_lat, cust_lon, sell_lat, sell_lon)`.
- `delivery_days` = delivered − purchase (in days); `estimate_days` = estimated − purchase (in days).
- `is_late` = 1 if delivered **after** estimated, else 0.
- From the purchase date make: `purchase_year, purchase_month, purchase_dayofweek, purchase_hour`,
  `is_weekend` (1 if dayofweek ≥ 5), and `log_freight` = `np.log1p(total_freight)`.

✍️ Write the prompt, paste the code, run it.

> 💡 If the prompt feels too big, split it: ask for the merges first, run it, then ask for the features.

In [ ]:
# 👇 Paste the AI's code for Stage 5 here.
# ── 1. Start from the cleaned orders base ────────────────────────────────────
df = orders_model.copy()                              # don't mutate the cleaned source

# ── 2. Merge dimension tables (all left so no rows are lost) ─────────────────
df = df.merge(customers,   on="customer_id", how="left")   # adds customer zip, city, state
df = df.merge(order_totals, on="order_id",   how="left")   # adds num_items, total_price, total_freight
df = df.merge(primary_pay,  on="order_id",   how="left")   # adds payment_type, payment_installments
df = df.merge(first_seller, on="order_id",   how="left")   # adds seller_id
df = df.merge(sellers,      on="seller_id",  how="left")   # adds seller zip, city, state

# ── 3. Attach lat/lon for customer zip ───────────────────────────────────────
df = df.merge(
    geo_lookup.rename(columns={"lat": "cust_lat", "lon": "cust_lon",
                                "geolocation_zip_code_prefix": "customer_zip_code_prefix"}),
    on="customer_zip_code_prefix", how="left",             # match on customer zip prefix
)

# ── 4. Attach lat/lon for seller zip ─────────────────────────────────────────
df = df.merge(
    geo_lookup.rename(columns={"lat": "sell_lat", "lon": "sell_lon",
                                "geolocation_zip_code_prefix": "seller_zip_code_prefix"}),
    on="seller_zip_code_prefix", how="left",               # match on seller zip prefix
)

# ── 5. Great-circle distance between customer and seller ─────────────────────
df["distance_km"] = haversine(                             # vectorised — no loop needed
    df["cust_lat"], df["cust_lon"],
    df["sell_lat"], df["sell_lon"],
)

# ── 6. Delivery timing features ──────────────────────────────────────────────
df["delivery_days"] = (                                    # actual days from purchase to delivery
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400                              # convert seconds → days (float)

df["estimate_days"] = (                                    # promised days from purchase to estimated
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# ── 7. Binary late flag ───────────────────────────────────────────────────────
df["is_late"] = (                                          # 1 if arrived after promised date
    df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]
).astype(int)

# ── 8. Purchase-timestamp calendar features ───────────────────────────────────
df["purchase_year"]      = df["order_purchase_timestamp"].dt.year         # e.g. 2017, 2018
df["purchase_month"]     = df["order_purchase_timestamp"].dt.month        # 1–12
df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek    # 0=Mon … 6=Sun
df["purchase_hour"]      = df["order_purchase_timestamp"].dt.hour         # 0–23
df["is_weekend"]         = (df["purchase_dayofweek"] >= 5).astype(int)    # 1 = Sat or Sun

# ── 9. Log-transformed freight (handles right skew) ──────────────────────────
df["log_freight"] = np.log1p(df["total_freight"])          # log(1 + x) safe when x = 0

# ── 10. Verify ───────────────────────────────────────────────────────────────
print(f"df shape        : {df.shape}")
print(f"is_late rate    : {df['is_late'].mean():.1%}")
print(f"distance_km NaN : {df['distance_km'].isna().sum()}")
print(f"delivery_days   : min={df['delivery_days'].min():.1f}  "
      f"median={df['delivery_days'].median():.1f}  "
      f"max={df['delivery_days'].max():.1f}")


✅ **Check:** `df` has ~96,470 rows; `df['is_late'].mean()` ≈ **0.08**; `df['distance_km']` averages ~600 km.

## Stage 6 — Add reviews + save the final file

📋 **Your task:** attach the review, keep the 21 needed columns, save `olist_clean.parquet`.

📦 **Put this in your prompt:**
- From `reviews`: one review per `order_id` (the first), keep `review_score`, `review_comment_message`.
  Merge it onto `df` (`how='left'`).
- Keep ONLY these 21 columns (give the AI this exact list):
  `order_id, customer_unique_id, customer_state, seller_state, purchase_year, purchase_month,`
  `purchase_dayofweek, purchase_hour, is_weekend, num_items, total_price, total_freight, log_freight,`
  `payment_type, payment_installments, distance_km, delivery_days, estimate_days, is_late,`
  `review_score, review_comment_message`
- Save as `olist_clean.parquet` with `index=False`. Print the shape and the `is_late` rate.

✍️ Write the prompt, paste the code, run it.

In [ ]:
# 👇 Paste the AI's code for Stage 6 here.
# ── 1. One review per order (take the first recorded review) ─────────────────
first_review = (
    reviews.sort_values("review_creation_date")           # earliest review floats to top
    .groupby("order_id", as_index=False)                  # one group per order
    .first()                                              # keep the earliest row
    [["order_id", "review_score", "review_comment_message"]]  # only the cols we need
)

# ── 2. Attach reviews to the master frame ────────────────────────────────────
df = df.merge(first_review, on="order_id", how="left")    # unreviewed orders → NaN score

# ── 3. Keep exactly the 21 required columns ──────────────────────────────────
KEEP = [
    "order_id", "customer_unique_id", "customer_state", "seller_state",
    "purchase_year", "purchase_month", "purchase_dayofweek", "purchase_hour",
    "is_weekend", "num_items", "total_price", "total_freight", "log_freight",
    "payment_type", "payment_installments", "distance_km", "delivery_days",
    "estimate_days", "is_late", "review_score", "review_comment_message",
]
df = df[KEEP]                                             # drop every other column

# ── 4. Save to parquet ───────────────────────────────────────────────────────
df.to_parquet("olist_clean.parquet", index=False)         # compact binary format, no row index

# ── 5. Verify ────────────────────────────────────────────────────────────────
print(f"df shape     : {df.shape}")                       # should be (96478, 21)
print(f"is_late rate : {df['is_late'].mean():.1%}")       # should be ≈ 7–8 %
print(f"review NaN   : {df['review_score'].isna().sum()}") # orders with no review

✅ **Check:** `olist_clean.parquet` has shape **(96470, 21)**, `is_late` rate ≈ **0.08**, reviews ≈ 99%.
**This is your final file. Do not lose it.**

## Stage 7 — Findings (write this yourself, NOT with AI)

Double-click and fill in (graded):

1. **Total rows in `olist_clean.parquet`:** ______
2. **`is_late` rate:** ______ (about 0.08)
3. **Three things you noticed while doing this:** ______ / ______ / ______
4. **One prompt that did NOT work the first time — what did you change to fix it?** ______

> Question 4 is the real lesson: getting a prompt to work on the second or third try **is** the skill.

## 📤 Submission

Put **your notebook AND `olist_clean.parquet`** in:
```
module-3/class_6/submissions/<YourTeamName>/
```

### 🎓 Grading (100 points)
| What we check | Points |
| --- | --- |
| Final file: 21 correct columns | 25 |
| Your prompts are clear (the 4 parts) — paste them in a text cell | 20 |
| Notebook runs top-to-bottom with no error | 20 |
| You **checked** each result against the ✅ targets | 15 |
| Findings filled in (incl. the fixed-prompt story) | 10 |
| `is_late` rate is 5–10% | 10 |

### Remember the recipe: **Context → Goal → What I have → Ask.** Give all 4, every time.

🏁 *You just made an AI build a real data pipeline for you — and you checked its work. That is the job.*